# Getting Started with Easy-EO

**Satellite imagery analysis in Python. From a raw Sentinel-2 scene to an NDVI map in a single expression.**

<a href="https://colab.research.google.com/github/tommyscodebase/easy-eo-tutorials/blob/main/00-getting-started/00-getting-started-with-easy-eo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>
<a href="https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID"><img src="https://img.shields.io/badge/YouTube-Watch%20the%20series-FF0000?logo=youtube&logoColor=white" alt="Watch the series on YouTube"></a>
<a href="https://github.com/Tommy-Burns/easy-eo"><img src="https://img.shields.io/badge/GitHub-easy--eo-181717?logo=github&logoColor=white" alt="easy-eo on GitHub"></a>
<a href="https://easy-eo.readthedocs.io"><img src="https://img.shields.io/badge/docs-easy--eo.readthedocs.io-8CA1AF?logo=readthedocs&logoColor=white" alt="Documentation"></a>
<a href="https://pypi.org/project/easy-eo/"><img src="https://img.shields.io/pypi/v/easy-eo.svg" alt="PyPI"></a>

---

This is the opening episode of the [Easy-EO series](https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID): a tour of what the library does, before we slow down and take each piece apart in later videos.

In the next few minutes you will:

1. **Load a real Sentinel-2 scene** from the hosted sample dataset, with no files to download by hand
2. **Inspect it**: CRS, shape, band count, and per-band statistics from one `.describe()` call
3. **Compute NDVI in one line**, and plot it
4. **Chain a whole workflow**: clip >> resample >> NDVI >> stretch >> plot: as a single expression
5. **Compare the built-in spectral indices** side by side, then stack them into one figure
6. **Search a live STAC catalog** and pull a real scene straight into your code

Everything except the last section runs offline once the sample data is cached.

### Before you run this
easy-eo must be installed. <a href="https://www.youtube.com/video"><img src="https://img.shields.io/badge/Watch%20this%20video-FF0000?logo=youtube&logoColor=white" alt="Watch this video"></a> or check the [the installation instructions](https://github.com/tommyscodebase/easy-eo-tutorials#setup) here

The final STAC section also needs the `stac` extra: `pip install "easy-eo[stac]"` (conda: `conda install -c conda-forge pystac-client planetary-computer`). Easy-EO requires Python 3.10+.


📓 All notebooks in the series: [tommyscodebase/easy-eo-tutorials](https://github.com/tommyscodebase/easy-eo-tutorials) · 🐛 Found a bug in the library? [Open an issue](https://github.com/Tommy-Burns/easy-eo/issues)

## **Running in Colab?** 
Uncomment and execute the cell below to install `easy-eo`

In [ ]:
# !pip install "easy-eo[stac]"

## 0. Setup (sample data) 

We use the bundled sample so this notebook needs no files of your own.

In [ ]:
from eeo.datasets import load_sample_dataset
from eeo import load_raster

sd = load_sample_dataset(prefetch=True)  # downloads once, cached after
scene = load_raster(sd.sentinel2_cog_stacked, band_names=["blue", "green", "red", "nir"])
scene.get_crs(), scene.get_shape(), scene.get_count()

Call `.describe()` on the dataset to get a description of the data.  
- `.describe(stats="approx")` or `.describe(stats=True)` gives you a description of the dataset together with a quick statistics of each band in the dataset
- `.describe(stats="exact")` gives you a description of the dataset together with an exact statistics of each band in the dataset

In [ ]:
scene.describe(stats=True)

## 2. NDVI in one line

In [ ]:
ndvi = scene.ndvi(red="red", nir="nir", name="My NDVI")

## Plot the NDVI

In [ ]:
ndvi.plot_raster(cmap="Greens")

## 3. Full chain operation in one expression

Clip to a bbox => resample => NDVI => stretch => plot, all in one expression.

In [ ]:
aoi_bbox = (380280.0, 5342790.0, 385400.0, 5347910.0)

(
    scene
    .clip_raster_with_bbox(aoi_bbox)
    .resample(scale_factor=2)
    .ndvi(red="red", nir="nir")
    .normalize_percentile(lower_percentile=2, upper_percentile=98)
    .plot_raster()
)

## 4. Example built-in indices, side by side

In [ ]:
fig_bands = dict(red="red", nir="nir", blue="blue")

indices = {
    "NDVI": scene.ndvi(red="red", nir="nir"),
    "NDWI": scene.ndwi(green="green", nir="nir"),
    "EVI":  scene.divide(10000).evi(red="red", blue="blue", nir="nir"),
    "SAVI": scene.savi(red="red", nir="nir"),
}

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(indices), figsize=(16, 4))
for ax, (name, result) in zip(axes, indices.items()):
    ax.imshow(result.get_band(1), cmap="RdYlGn")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Each index above is its own single-band dataset - stack them into one
# multiband EEORasterDataset so we can hand the whole panel to plot_raster
# instead of driving Matplotlib by hand.
stacked = indices["NDVI"].stack(
    [indices["NDWI"], indices["EVI"], indices["SAVI"]],
    names=["NDVI", "NDWI", "EVI", "SAVI"],
)

stacked.plot_raster(cmap="RdYlGn", figsize=(8, 8), title="NDVI · NDWI · EVI · SAVI", colorbar=True, ncols=2)

## 5. Get real satellite scenes right in your code (needs network + the `stac` extra)

`pip install "easy-eo[stac]"` if you haven't already

In [ ]:
import eeo

results = eeo.stac_search(
    "sentinel-2-l2a",
    bbox=(11.0, 46.5, 11.2, 46.7),
    datetime="2023-06-01/2023-08-30",
    cloud_cover=20,
    limit=1,
)

## Compute NDVI and plot

In [ ]:
live_scene = results[0].load(["B04", "B08"])
live_ndvi = live_scene.ndvi(red="B04", nir="B08", name="NDVI")
live_ndvi.plot_raster(cmap="Greens", colorbar=True)

## Compute NDWI and plot

In [ ]:
live_scene = results[0].load(["B03", "B08"])
live_ndwi = live_scene.ndwi(nir="B08", green="B03", name="NDWI")
live_ndwi.plot_raster(cmap="Blues", colorbar=True)